# Object Pipeline
Notebook này huấn luyện riêng object classifier với tên file và biến rõ ràng theo object pipeline.

In [ ]:
import os
import sys
import json
import random
from pathlib import Path

import torch
from torch.utils.data import DataLoader

current_dir = os.path.abspath(os.getcwd())
project_root = os.path.dirname(current_dir) if current_dir.endswith("notebooks") else current_dir
if project_root not in sys.path:
    sys.path.insert(0, project_root)
os.chdir(project_root)

from src.utils.config_loader import load_task_configs, print_config
from src.data.download import download_and_extract_metadata, download_vg_images, verify_download
from src.data.preprocessing import build_object_attribute_vocab_and_splits
from src.features.object_feature_extractor import extract_object_features
from src.data.object_dataset import build_object_datasets
from src.models.object import ObjectClassifier
from src.training.object_trainer import ObjectTrainer
from src.evaluation.metrics import compute_classification_metrics
from src.utils.memory import cleanup_cuda_memory

base_config = load_task_configs("configs/config.yaml")
object_config = load_task_configs("configs/config.yaml", "configs/object_config.yaml")

project_root_path = Path.cwd()
raw_dir = project_root_path / base_config.paths.raw_dir
image_dir = project_root_path / base_config.dataset.image_dir
processed_dir = project_root_path / object_config.dataset.processed_dir
feature_dir = processed_dir / object_config.dataset.feature_cache_dir

device = "cuda" if str(base_config.device).lower().startswith("cuda") and torch.cuda.is_available() else "cpu"

def normalize_input_mode(mode):
    mode = str(mode).lower().strip()
    aliases = {
        "grayscale": "gray",
        "grey": "gray",
        "edge": "contour",
        "edges": "contour",
    }
    mode = aliases.get(mode, mode)
    if mode not in {"rgb", "gray", "contour"}:
        raise ValueError("input mode phải là 'rgb', 'gray', hoặc 'contour'")
    return mode


def require_files(paths, label):
    missing = [str(path) for path in paths if not Path(path).exists()]
    if missing:
        raise FileNotFoundError(f"Thiếu {label}: {missing}")


def ensure_nonempty_cache(cache_path):
    cache_path = Path(cache_path)
    if not cache_path.exists():
        return False
    if cache_path.stat().st_size == 0:
        return False
    try:
        cache = torch.load(cache_path, map_location="cpu")
        return bool(cache)
    except Exception:
        return False


def cache_output(split_name, input_mode):
    suffix = None if input_mode == "rgb" else input_mode
    filename = f"{split_name}_features.pt" if suffix is None else f"{split_name}_{suffix}_features.pt"
    return feature_dir / filename


def collect_split_image_ids(processed_root, split_names):
    image_ids = set()
    for split_name in split_names:
        annotation_file = Path(processed_root) / split_name / "annotations.json"
        if not annotation_file.exists():
            raise FileNotFoundError(f"Thiếu annotation: {annotation_file}")

        with open(annotation_file, "r", encoding="utf-8") as f:
            raw = json.load(f)

        samples = raw if isinstance(raw, list) else raw.get("samples", [])
        image_ids.update(sample["image_id"] for sample in samples)

    return sorted(image_ids)


def ensure_images_for_splits(pipeline_name, processed_root, split_names):
    image_ids = collect_split_image_ids(processed_root, split_names)
    missing_ids = [img_id for img_id in image_ids if not (image_dir / f"{img_id}.jpg").exists()]

    print(f"[{pipeline_name}] Tổng ảnh tham chiếu: {len(image_ids)} | thiếu local: {len(missing_ids)}")
    if missing_ids:
        print(f"[{pipeline_name}] Đang tải bổ sung ảnh còn thiếu...")
        downloaded = download_vg_images(missing_ids, image_dir=str(image_dir))
        if len(downloaded) < len(missing_ids):
            print(f"[Warning] {pipeline_name}: còn {len(missing_ids) - len(downloaded)} ảnh chưa tải được.")


download_data = bool(base_config.pipeline.download_data)
strict_sample_mode = bool(base_config.sampling.strict_mode)
sample_size = int(base_config.sampling.sample_size)
sample_seed = int(base_config.sampling.seed)
image_download_mode = "none" if strict_sample_mode else str(base_config.sampling.image_download_mode)
pre_extract_features = bool(base_config.pipeline.pre_extract_features)

feature_batch_size = int(base_config.feature_extraction.batch_size)
feature_resize_size = int(base_config.feature_extraction.resize_size)
feature_crop_size = int(base_config.feature_extraction.crop_size)
feature_mean = [float(x) for x in base_config.image.mean]
feature_std = [float(x) for x in base_config.image.std]
full_split_ratios = (
    float(base_config.split.train),
    float(base_config.split.val),
    float(base_config.split.test),
)
preprocess_split_ratios = (
    tuple(float(x) for x in base_config.sampling.split_ratios)
    if strict_sample_mode
    else full_split_ratios
)
max_samples = None if base_config.sampling.max_samples is None else int(base_config.sampling.max_samples)

object_learnable_backbone = bool(object_config.backbone.learnable_backbone)
object_use_cache = bool(object_config.dataset.use_feature_cache) and not object_learnable_backbone
object_input_mode = normalize_input_mode(object_config.dataset.object_input_mode)
object_batch_size = int(object_config.training.batch_size)
object_lr = float(object_config.training.lr)
object_max_epochs = int(object_config.training.max_epochs)
object_roi_size = int(base_config.image.roi_size)
object_hidden_dim = int(object_config.model.object_hidden_dim)
object_dropout = float(object_config.model.object_dropout)
object_num_layers = int(object_config.model.object_num_layers)

print_config(base_config)
print_config(object_config)
print(f"Project root: {project_root_path}")
print(f"Using device: {device}")
print("Pipeline: Object")
print(f"Strict sample mode: {strict_sample_mode}")
print(f"Sample size: {sample_size} | split ratios: {preprocess_split_ratios} | seed: {sample_seed}")
print(f"Processed root: {processed_dir}")
print(f"Feature extraction batch size: {feature_batch_size}")
print(f"Object learnable backbone: {object_learnable_backbone}")
print(f"Object input mode: {object_input_mode}")
print(f"Object batch size: {object_batch_size}, lr: {object_lr}, max_epochs: {object_max_epochs}")

CONFIGURATION
project_name: ImageCaptioningE2E
seed: 42
device: cpu
paths:
  data_dir: data
  raw_dir: data/raw
  processed_dir: data/processed
  feature_dir: data/features
  checkpoint_dir: checkpoints
  log_dir: logs
dataset:
  repo_id: ranjaykrishna/visual_genome
  cache_dir: data/raw/hf_cache
  image_dir: data/raw/images
image:
  roi_size: 224
  mean:
  - 0.485
  - 0.456
  - 0.406
  std:
  - 0.229
  - 0.224
  - 0.225
split:
  train: 0.7
  val: 0.15
  test: 0.15
training:
  batch_size: 512
  num_workers: 4
  pin_memory: true
  max_epochs: 30
  early_stopping_patience: 5
  gradient_clip_val: 1.0
  log_every_n_steps: 0
optimizer:
  name: adamw
  lr: 0.0001
  weight_decay: 0.0001
scheduler:
  name: cosine
  warmup_epochs: 2
logging:
  use_tensorboard: true
  use_wandb: false
  wandb_project: vg-caption
pipeline:
  training_mode: all
  download_data: true
  pre_extract_features: true
sampling:
  strict_mode: true
  sample_size: 200
  split_ratios:
  - 0.8
  - 0.1
  - 0.1
  seed: 42
  im

In [ ]:
# Load or verify raw data
if download_data:
    print('Downloading metadata for the object pipeline...')
    download_and_extract_metadata(raw_dir=str(raw_dir), keep_zip=False)

    raw_status = verify_download(raw_dir=str(raw_dir))
    missing_metadata = [name for name, ok in raw_status.items() if not ok]
    if missing_metadata:
        raise RuntimeError(f'Thiếu metadata sau khi tải: {missing_metadata}')

    with open(raw_dir / 'image_data.json', 'r', encoding='utf-8') as f:
        image_data = json.load(f)

    if image_download_mode == 'sample':
        sample_ids = [img['image_id'] for img in image_data[:sample_size]]
        print(f'Bắt đầu tải bộ sample {len(sample_ids)} ảnh...')
        downloaded_images = download_vg_images(sample_ids, image_dir=str(image_dir))
        if not downloaded_images:
            raise RuntimeError('Không tải được ảnh mẫu nào.')
    elif image_download_mode == 'all':
        all_ids = [img['image_id'] for img in image_data]
        print(f'Bắt đầu tải toàn bộ {len(all_ids)} ảnh...')
        downloaded_images = download_vg_images(all_ids, image_dir=str(image_dir))
        if not downloaded_images:
            raise RuntimeError('Không tải được ảnh nào.')
    elif image_download_mode == 'none':
        print('Bỏ qua tải ảnh theo cấu hình.')
    else:
        raise ValueError("IMAGE_DOWNLOAD_MODE phải là 'none', 'sample', hoặc 'all'.")
else:
    print('Bỏ qua tải mới, chỉ kiểm tra dữ liệu hiện có...')
    raw_status = verify_download(raw_dir=str(raw_dir))
    missing_metadata = [name for name, ok in raw_status.items() if not ok]
    if missing_metadata:
        raise RuntimeError(
            'Thiếu dữ liệu RAW cần thiết: ' + ', '.join(missing_metadata) +
            '. Hãy bật DOWNLOAD_DATA = True hoặc đặt đúng thư mục data/raw.'
        )

# Build strict sample image ids if required
if strict_sample_mode:
    image_data_file = raw_dir / 'image_data.json'
    require_files([image_data_file], 'image_data.json')

    with open(image_data_file, 'r', encoding='utf-8') as f:
        image_data = json.load(f)

    all_image_ids = [img['image_id'] for img in image_data]
    sample_count = min(sample_size, len(all_image_ids))
    sample_image_ids = random.Random(sample_seed).sample(all_image_ids, sample_count)
    print(f'Đã chọn trước {len(sample_image_ids)} image_id cho sample strict.')
else:
    sample_image_ids = None
    print('Bỏ qua strict sample; sẽ dùng toàn bộ dữ liệu theo split mặc định.')

# Build object/attribute vocabularies and split files
build_object_attribute_vocab_and_splits(
    raw_dir=str(raw_dir),
    processed_dir=str(processed_dir),
    max_objects=int(object_config.labels.max_objects),
    max_attributes=int(object_config.labels.max_attributes),
    sample_image_ids=sample_image_ids if strict_sample_mode else None,
    split_by_image_id=strict_sample_mode,
    split_ratios=preprocess_split_ratios,
    seed=sample_seed,
)

require_files(
    [
        processed_dir / 'object_vocab.json',
        processed_dir / 'attribute_vocab.json',
        processed_dir / 'train' / 'annotations.json',
        processed_dir / 'val' / 'annotations.json',
        processed_dir / 'test' / 'annotations.json',
    ],
    'Object pipeline processed files',
)

# Optional feature extraction when the backbone is not learnable inside the model
if pre_extract_features and object_use_cache:
    feature_dir.mkdir(parents=True, exist_ok=True)
    ensure_images_for_splits('Object', processed_dir, ['train', 'val', 'test'])

    print('Extracting object features...')
    for split_name in ['train', 'val', 'test']:
        object_output = cache_output(split_name, object_input_mode)
        extract_object_features(
            annotation_file=str(processed_dir / split_name / 'annotations.json'),
            image_dir=str(image_dir),
            output_file=str(object_output),
            backbone=str(object_config.backbone.name),
            pretrained=bool(object_config.backbone.pretrained),
            batch_size=feature_batch_size,
            device=device,
            resize_size=feature_resize_size,
            crop_size=feature_crop_size,
            mean=feature_mean,
            std=feature_std,
            input_mode=object_input_mode,
        )
        if not ensure_nonempty_cache(object_output):
            raise RuntimeError(f'Object feature cache rỗng: {object_output}')
else:
    print('Bỏ qua feature extraction theo cấu hình PRE_EXTRACT_FEATURES hoặc learnable backbone.')

# Build datasets and train the object classifier
object_train_ds, object_val_ds, object_test_ds = build_object_datasets(
    processed_dir=str(processed_dir),
    image_dir=str(image_dir),
    roi_size=object_roi_size,
    use_feature_cache=object_use_cache,
    feature_cache_dir=str(feature_dir),
    input_mode=object_input_mode,
    max_samples=max_samples,
    train_horizontal_flip_p=float(object_config.augmentation.random_horizontal_flip),
    train_color_jitter=bool(object_config.augmentation.color_jitter.enabled),
    train_brightness=float(object_config.augmentation.color_jitter.brightness),
    train_contrast=float(object_config.augmentation.color_jitter.contrast),
    train_saturation=float(object_config.augmentation.color_jitter.saturation),
    train_hue=float(object_config.augmentation.color_jitter.hue),
    train_random_erasing_p=float(object_config.augmentation.random_erasing_p),
    train_resize_delta=int(object_config.augmentation.resize_delta),
    mean=feature_mean,
    std=feature_std,
)

object_model = ObjectClassifier(
    num_classes=object_train_ds.num_objects,
    feature_dim=int(object_config.backbone.feature_dim),
    hidden_dim=object_hidden_dim,
    dropout=object_dropout,
    num_layers=object_num_layers,
    backbone_name=str(object_config.backbone.name) if object_learnable_backbone else None,
    pretrained=bool(object_config.backbone.pretrained),
    freeze_backbone=bool(object_config.backbone.freeze_backbone),
    learnable_backbone=object_learnable_backbone,
    device=device,
)

object_optimizer = torch.optim.AdamW(
    object_model.parameters(),
    lr=object_lr,
    weight_decay=float(base_config.optimizer.weight_decay),
)

object_train_loader = DataLoader(
    object_train_ds,
    batch_size=object_batch_size,
    shuffle=True,
    num_workers=int(base_config.training.num_workers),
    pin_memory=bool(base_config.training.pin_memory and device == 'cuda'),
)
object_val_loader = DataLoader(
    object_val_ds,
    batch_size=object_batch_size,
    shuffle=False,
    num_workers=int(base_config.training.num_workers),
    pin_memory=bool(base_config.training.pin_memory and device == 'cuda'),
)
object_test_loader = DataLoader(
    object_test_ds,
    batch_size=object_batch_size,
    shuffle=False,
    num_workers=int(base_config.training.num_workers),
    pin_memory=bool(base_config.training.pin_memory and device == 'cuda'),
)

object_trainer = ObjectTrainer(
    model=object_model,
    train_loader=object_train_loader,
    val_loader=object_val_loader,
    optimizer=object_optimizer,
    use_auto_class_weights=True,
    freeze_backbone=bool(object_config.backbone.freeze_backbone),
    freeze_epochs=int(object_config.backbone.freeze_epochs),
    max_epochs=object_max_epochs,
    early_stopping_patience=int(base_config.training.early_stopping_patience),
    gradient_clip_val=float(base_config.training.gradient_clip_val),
    log_every_n_steps=int(base_config.training.log_every_n_steps),
    device=device,
    use_amp=(device == 'cuda'),
)

print('Bắt đầu training object pipeline...')
object_val_metrics = object_trainer.train()
object_best_checkpoint = object_trainer.checkpoint_manager.get_best_checkpoint_path()
print('Object pipeline completed.')
print(f'Best object checkpoint: {object_best_checkpoint}')

object_model = object_trainer.object_model
object_test_metrics = None
object_logits_list = []
object_targets_list = []
with torch.no_grad():
    for batch in object_test_loader:
        batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
        features = batch.get('feature')
        if features is None:
            features = batch.get('object_feature')
        if features is None:
            features = batch.get('image')
        if features is None:
            features = batch.get('object_image')
        logits = object_model(features)
        object_logits_list.append(logits.cpu())
        object_targets_list.append(batch['object_label'].cpu())

object_test_metrics = compute_classification_metrics(
    torch.cat(object_logits_list),
    torch.cat(object_targets_list),
    num_classes=object_train_ds.num_objects,
)
print('Object test metrics:')
print(object_test_metrics)

[Skip] objects.json đã tồn tại.
[Skip] attributes.json đã tồn tại.
[Skip] relationships.json đã tồn tại.
[Skip] image_data.json đã tồn tại.

--- Kiểm tra dữ liệu RAW ---
  ✅  objects.json (412.6 MB)
  ✅  attributes.json (462.6 MB)
  ✅  relationships.json (709.6 MB)
  ✅  image_data.json (17.6 MB)
Bỏ qua tải ảnh theo cấu hình.
Đã chọn trước 200 image_id cho sample strict.
--- [Task 1] Bắt đầu Preprocessing ---
Đang load objects file...
Đang load attributes file...
[Task 1] Giới hạn preprocessing trên 200 image_id đã chọn.
Đang đếm tần suất object & attributes...


Parsing: 100%|██████████| 108077/108077 [00:04<00:00, 22361.98it/s]


Đang mapping dữ liệu sang integer indices...
Tổng hợp được 5342 samples hợp lệ.
--- Hoàn tất Task 1 Preprocessing ---
Bỏ qua feature extraction theo cấu hình PRE_EXTRACT_FEATURES hoặc learnable backbone.
[ObjectAttributeDataset] Loaded 4494 annotations từ c:\Users\caoha\OneDrive\Desktop\study\Ky2\CV\prj\data\processed\object\train\annotations.json
ObjectAttributeDataset [train]: 4494 ROIs | 158 images | 301 objects | 201 attributes | cache=❌
[ObjectAttributeDataset] Loaded 444 annotations từ c:\Users\caoha\OneDrive\Desktop\study\Ky2\CV\prj\data\processed\object\val\annotations.json
ObjectAttributeDataset [val]: 444 ROIs | 20 images | 301 objects | 201 attributes | cache=❌
[ObjectAttributeDataset] Loaded 404 annotations từ c:\Users\caoha\OneDrive\Desktop\study\Ky2\CV\prj\data\processed\object\test\annotations.json
ObjectAttributeDataset [test]: 404 ROIs | 20 images | 301 objects | 201 attributes | cache=❌
2026-04-06 09:40:16 [INFO] TensorBoard logging to: logs\tensorboard\vg_caption
202

In [ ]:
cleanup_cuda_memory(note='Object pipeline notebook finished')